# Hybrid Retrieval with Structured Outputs

You are going to build a hybrid retrieval system that combines two ideas:

1. **Structured outputs** to convert a natural language query into exact SQL filters.
2. **Vector search** to rank the filtered results by semantic similarity.

You will use a toy e-commerce dataset stored entirely in SQLite, so there is nothing to install beyond the OpenAI SDK.

By the end of this notebook you will understand why hybrid retrieval works better than either approach alone, and you will have a working pattern you can adapt to real projects.

In [ ]:
# DEPENDENCY: pip install -q --upgrade openai numpy sqlite3
# (packages should be pre-installed in venv before running this script)

In [1]:
from getpass import getpass
import sqlite3
import json
import numpy as np
from openai import OpenAI
import os
from pydantic import BaseModel
from typing import Optional


# API Key Setup - uses environment variable or prompts for input
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")


client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"


In [2]:
MODEL = "gpt-5-mini"

## 1. Build the Toy Dataset in SQLite

You are creating a small e-commerce product catalog with 15 products across four categories.

Each product has a name, category, price, and a short description. You store everything in a single SQLite database, including the embedding vectors.

In [3]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE products (
        id INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        description TEXT NOT NULL,
        in_stock INTEGER NOT NULL DEFAULT 1,
        embedding BLOB
    )
""")

products = [
    ("Wireless Bluetooth Earbuds", "electronics", 29.99, "Compact earbuds with noise cancellation and 8 hour battery life", 1),
    ("USB-C Charging Cable 6ft", "electronics", 12.99, "Braided nylon cable for fast charging phones and tablets", 1),
    ("Portable Bluetooth Speaker", "electronics", 45.00, "Waterproof speaker with deep bass and 12 hour playtime", 1),
    ("Mechanical Keyboard", "electronics", 89.99, "RGB backlit keyboard with cherry red switches for typing and gaming", 0),
    ("Noise Cancelling Headphones", "electronics", 149.99, "Over ear headphones with active noise cancellation and premium sound", 1),
    ("Cotton Crew Neck T-Shirt", "clothing", 18.50, "Soft 100 percent cotton tee available in multiple colors", 1),
    ("Slim Fit Jeans", "clothing", 49.99, "Stretch denim jeans with a modern slim fit", 1),
    ("Waterproof Rain Jacket", "clothing", 75.00, "Lightweight jacket with sealed seams for rainy weather", 1),
    ("Running Shoes", "clothing", 95.00, "Breathable mesh shoes with cushioned soles for long runs", 0),
    ("Stainless Steel Water Bottle", "home", 22.00, "Double wall insulated bottle that keeps drinks cold for 24 hours", 1),
    ("Scented Soy Candle", "home", 14.99, "Hand poured lavender candle with a 40 hour burn time", 1),
    ("Bamboo Cutting Board", "home", 19.99, "Eco friendly cutting board with juice groove and easy grip handles", 1),
    ("Organic Green Tea", "food", 8.99, "Japanese sencha green tea with a smooth and mild flavor", 1),
    ("Dark Chocolate Bar 72%", "food", 4.50, "Single origin cacao bar with rich bittersweet flavor", 1),
    ("Mixed Nuts Snack Pack", "food", 11.99, "Roasted almonds cashews and walnuts with sea salt", 1),
]

for name, category, price, description, in_stock in products:
    cursor.execute(
        "INSERT INTO products (name, category, price, description, in_stock) VALUES (?, ?, ?, ?, ?)",
        (name, category, price, description, in_stock)
    )

conn.commit()
print(f"Inserted {len(products)} products into SQLite")

# Quick look at the data
cursor.execute("SELECT id, name, category, price, in_stock FROM products")
for row in cursor.fetchall():
    stock_label = "in stock" if row[4] else "out of stock"
    print(f"  [{row[0]:2d}] ${row[3]:>7.2f}  {row[2]:<12s}  {row[1]}  ({stock_label})")

Inserted 15 products into SQLite
  [ 1] $  29.99  electronics   Wireless Bluetooth Earbuds  (in stock)
  [ 2] $  12.99  electronics   USB-C Charging Cable 6ft  (in stock)
  [ 3] $  45.00  electronics   Portable Bluetooth Speaker  (in stock)
  [ 4] $  89.99  electronics   Mechanical Keyboard  (out of stock)
  [ 5] $ 149.99  electronics   Noise Cancelling Headphones  (in stock)
  [ 6] $  18.50  clothing      Cotton Crew Neck T-Shirt  (in stock)
  [ 7] $  49.99  clothing      Slim Fit Jeans  (in stock)
  [ 8] $  75.00  clothing      Waterproof Rain Jacket  (in stock)
  [ 9] $  95.00  clothing      Running Shoes  (out of stock)
  [10] $  22.00  home          Stainless Steel Water Bottle  (in stock)
  [11] $  14.99  home          Scented Soy Candle  (in stock)
  [12] $  19.99  home          Bamboo Cutting Board  (in stock)
  [13] $   8.99  food          Organic Green Tea  (in stock)
  [14] $   4.50  food          Dark Chocolate Bar 72%  (in stock)
  [15] $  11.99  food          Mixed Nuts S

## 2. Generate and Store Embeddings

You now generate an embedding for each product by combining its name and description into a single text string.

You store the embedding as a binary blob directly in the SQLite table. This is simple and works well for small datasets. For production systems you would use a dedicated vector database, but for learning the concept SQLite is perfect.

In [4]:
def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a batch of texts using OpenAI."""
    response = client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL
    )
    return [item.embedding for item in response.data]

def embedding_to_bytes(embedding: list[float]) -> bytes:
    """Convert a float list to bytes for SQLite storage."""
    return np.array(embedding, dtype=np.float32).tobytes()

def bytes_to_embedding(blob: bytes) -> np.ndarray:
    """Convert bytes back to a numpy array."""
    return np.frombuffer(blob, dtype=np.float32)

# Build the text to embed for each product
cursor.execute("SELECT id, name, description FROM products")
rows = cursor.fetchall()

texts = [f"{name}: {description}" for _, name, description in rows]
ids = [row[0] for row in rows]

print(f"Generating embeddings for {len(texts)} products...")
embeddings = get_embeddings(texts)
print(f"Each embedding has {len(embeddings[0])} dimensions")

# Store embeddings back in SQLite
for product_id, embedding in zip(ids, embeddings):
    cursor.execute(
        "UPDATE products SET embedding = ? WHERE id = ?",
        (embedding_to_bytes(embedding), product_id)
    )

conn.commit()
print("Embeddings stored in SQLite")

Generating embeddings for 15 products...
Each embedding has 1536 dimensions
Embeddings stored in SQLite


## 3. Pure Vector Search

You now have embeddings stored alongside your product data. You can search by embedding a query and comparing it to every product using cosine similarity.

This approach finds semantically similar products, but it ignores structured constraints like price or category.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)

def vector_search(query: str, top_k: int = 5) -> list[dict]:
    """Search all products by semantic similarity to the query."""
    query_embedding = get_embeddings([query])[0]
    query_vec = np.array(query_embedding, dtype=np.float32)
    
    cursor.execute("SELECT id, name, category, price, in_stock, embedding FROM products")
    results = []
    for row in cursor.fetchall():
        product_vec = bytes_to_embedding(row[5])
        similarity = cosine_similarity(query_vec, product_vec)
        results.append({
            "id": row[0],
            "name": row[1],
            "category": row[2],
            "price": row[3],
            "in_stock": bool(row[4]),
            "similarity": similarity
        })
    
    results.sort(key=lambda x: x["similarity"], reverse=True)
    return results[:top_k]

# Test it
query = "something to listen to music with"
print(f"Query: '{query}'\n")
for r in vector_search(query):
    stock = "in stock" if r["in_stock"] else "out of stock"
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}  ({stock})")

Query: 'hydraton'

  [10] sim=0.189  $  22.00  Stainless Steel Water Bottle  (in stock)
  [ 3] sim=0.174  $  45.00  Portable Bluetooth Speaker  (in stock)
  [ 8] sim=0.163  $  75.00  Waterproof Rain Jacket  (in stock)
  [ 9] sim=0.145  $  95.00  Running Shoes  (out of stock)
  [15] sim=0.105  $  11.99  Mixed Nuts Snack Pack  (in stock)


You can see that vector search returns semantically relevant results. But what if you only want items under $50 that are in stock? The vector search has no way to enforce that. It just ranks by meaning.

You need a way to extract those constraints from the query automatically.

## 4. Structured Outputs for Query Understanding

You now use OpenAI structured outputs to convert a natural language query into a set of SQL filters.

You define a Pydantic model that describes the filters you support. The model will always return valid JSON matching this schema, so you never have to worry about parsing errors or missing fields.

In [7]:
class ProductFilter(BaseModel):
    """Structured filters extracted from a natural language product search query."""
    category: Optional[str] = None        # electronics, clothing, home, food
    max_price: Optional[float] = None      # upper price limit
    min_price: Optional[float] = None      # lower price limit
    in_stock_only: bool = True              # default to only showing available items
    search_text: str                        # the semantic part of the query for vector search

print("ProductFilter schema:")
print(json.dumps(ProductFilter.model_json_schema(), indent=2))

ProductFilter schema:
{
  "description": "Structured filters extracted from a natural language product search query.",
  "properties": {
    "category": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Category"
    },
    "max_price": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Max Price"
    },
    "min_price": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Min Price"
    },
    "in_stock_only": {
      "default": true,
      "title": "In Stock Only",
      "type": "boolean"
    },
    "search_text": {
      "title": "Search Text",
      "type": "string"
    }
  },
  "required": [
    "search_text"
  ],
  "title": "ProductFilter",
  "type": "object"


In [8]:
def parse_query(user_query: str) -> ProductFilter:
    """Use structured outputs to extract filters from a natural language query."""
    response = client.responses.parse(
        model=MODEL,
        instructions=(
            # Add the database schema to the prompt:
            # SQLlite - sqllite_master (type = 'table')
            "You extract product search filters from a user query. "
            "The available categories are: electronics, clothing, home, food. "
            "Extract any price constraints, category, and stock preferences. "
            "Put the semantic search intent into search_text. "
            "If the user does not mention a category, leave it as null. "
            "If the user does not mention price, leave min_price and max_price as null."
        ),
        input=user_query,
        text_format=ProductFilter,
    )
    return response.output_parsed

# Test with a few queries
test_queries = [
    "I want affordable headphones under $50",
    "Show me food items",
    "Something warm for rainy days, even if it is out of stock",
    "Premium audio gear between $100 and $200",
]

for q in test_queries:
    filters = parse_query(q)
    print(f"Query: '{q}'")
    print(f"  => {filters}")
    print()

Query: 'I want affordable headphones under $50'
  => category='electronics' max_price=50.0 min_price=None in_stock_only=False search_text='affordable headphones under $50'

Query: 'Show me food items'
  => category='food' max_price=None min_price=None in_stock_only=False search_text='food items'

Query: 'Something warm for rainy days, even if it is out of stock'
  => category=None max_price=None min_price=None in_stock_only=False search_text='warm items for rainy days (rain jacket, waterproof insulated coat, fleece, thermal layers, rain boots)'

Query: 'Premium audio gear between $100 and $200'
  => category='electronics' max_price=200.0 min_price=100.0 in_stock_only=False search_text='premium audio gear'



You now have a reliable way to turn messy human language into clean, typed filters.

Notice that the model extracted the semantic intent into `search_text` while pulling out the structured constraints separately. This is the key insight: you let SQL handle exact filtering and vector search handle meaning.

## 5. SQL Filtering

You now build a function that turns the structured filters into a SQL WHERE clause. This narrows the candidate set before you do any vector comparison.

In [9]:
def build_sql_filter(filters: ProductFilter) -> tuple[str, list]:
    """Convert ProductFilter into a SQL WHERE clause with parameters."""
    conditions = []
    params = []
    
    if filters.category:
        conditions.append("category = ?")
        params.append(filters.category.lower())
    
    if filters.max_price is not None:
        conditions.append("price <= ?")
        params.append(filters.max_price)
    
    if filters.min_price is not None:
        conditions.append("price >= ?")
        params.append(filters.min_price)
    
    if filters.in_stock_only:
        conditions.append("in_stock = 1")
    
    where_clause = " AND ".join(conditions) if conditions else "1=1"
    return where_clause, params

def sql_filter_search(filters: ProductFilter) -> list[dict]:
    """Return products matching the SQL filters."""
    where_clause, params = build_sql_filter(filters)
    query = f"SELECT id, name, category, price, in_stock, embedding FROM products WHERE {where_clause}"
    
    cursor.execute(query, params)
    results = []
    for row in cursor.fetchall():
        results.append({
            "id": row[0],
            "name": row[1],
            "category": row[2],
            "price": row[3],
            "in_stock": bool(row[4]),
            "embedding": bytes_to_embedding(row[5])
        })
    return results

# Test: electronics under $50 that are in stock
test_filter = ProductFilter(
    category="electronics",
    max_price=50.0,
    in_stock_only=True,
    search_text="headphones"
)

where_clause, params = build_sql_filter(test_filter)
print(f"SQL WHERE: {where_clause}")
print(f"Parameters: {params}")
print()

filtered = sql_filter_search(test_filter)
print(f"Found {len(filtered)} products matching filters:")
for p in filtered:
    print(f"  [{p['id']:2d}] ${p['price']:>7.2f}  {p['name']}")

SQL WHERE: category = ? AND price <= ? AND in_stock = 1
Parameters: ['electronics', 50.0]

Found 3 products matching filters:
  [ 1] $  29.99  Wireless Bluetooth Earbuds
  [ 2] $  12.99  USB-C Charging Cable 6ft
  [ 3] $  45.00  Portable Bluetooth Speaker


## 6. Hybrid Retrieval: SQL Filters + Vector Ranking

You now combine both approaches. The pipeline works in three steps:

1. Parse the user query into structured filters using the LLM.
2. Use SQL to narrow the candidate set based on category, price, and stock.
3. Rank the filtered candidates by vector similarity to the semantic search text.

This gives you the best of both worlds: exact constraints are enforced perfectly by SQL, and semantic ranking surfaces the most relevant matches.

In [11]:
def hybrid_search(user_query: str, top_k: int = 5, verbose: bool = True) -> list[dict]:
    """Full hybrid retrieval pipeline: parse, filter, rank."""
    
    # Step 1: Parse query into structured filters
    filters = parse_query(user_query)
    if verbose:
        print(f"Parsed filters:")
        print(f"  category:      {filters.category}")
        print(f"  price range:   {filters.min_price} to {filters.max_price}")
        print(f"  in stock only: {filters.in_stock_only}")
        print(f"  search text:   '{filters.search_text}'")
        print()
    
    # Step 2: SQL filter
    candidates = sql_filter_search(filters)
    if verbose:
        where_clause, params = build_sql_filter(filters)
        print(f"SQL filter: WHERE {where_clause}")
        print(f"Candidates after filtering: {len(candidates)}")
        print()
    
    if not candidates:
        if verbose:
            print("No products match the filters.")
        return []
    
    # Step 3: Vector rank
    query_embedding = get_embeddings([filters.search_text])[0]
    query_vec = np.array(query_embedding, dtype=np.float32)
    
    for candidate in candidates:
        candidate["similarity"] = cosine_similarity(query_vec, candidate["embedding"])
    
    candidates.sort(key=lambda x: x["similarity"], reverse=True)
    
    # Clean up: remove raw embedding from output
    for c in candidates:
        del c["embedding"]
    
    return candidates[:top_k]

In [12]:
query = "I want affordable headphones under $50"
print(f"Query: '{query}'")
print("=" * 60)
results = hybrid_search(query)
print("Results:")
for r in results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

Query: 'I want affordable headphones under $50'
Parsed filters:
  category:      electronics
  price range:   None to 50.0
  in stock only: False
  search text:   'affordable headphones'

SQL filter: WHERE category = ? AND price <= ?
Candidates after filtering: 3

Results:
  [ 1] sim=0.473  $  29.99  Wireless Bluetooth Earbuds
  [ 3] sim=0.357  $  45.00  Portable Bluetooth Speaker
  [ 2] sim=0.192  $  12.99  USB-C Charging Cable 6ft


In [13]:
query = "healthy snacks and drinks"
print(f"Query: '{query}'")
print("=" * 60)
results = hybrid_search(query)
print("Results:")
for r in results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

Query: 'healthy snacks and drinks'
Parsed filters:
  category:      food
  price range:   None to None
  in stock only: False
  search text:   'healthy snacks and drinks'

SQL filter: WHERE category = ?
Candidates after filtering: 3

Results:
  [15] sim=0.423  $  11.99  Mixed Nuts Snack Pack
  [13] sim=0.227  $   8.99  Organic Green Tea
  [14] sim=0.190  $   4.50  Dark Chocolate Bar 72%


In [14]:
query = "something cozy for my home under $25"
print(f"Query: '{query}'")
print("=" * 60)
results = hybrid_search(query)
print("Results:")
for r in results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

Query: 'something cozy for my home under $25'
Parsed filters:
  category:      home
  price range:   None to 25.0
  in stock only: False
  search text:   'cozy items for my home'

SQL filter: WHERE category = ? AND price <= ?
Candidates after filtering: 3

Results:
  [11] sim=0.232  $  14.99  Scented Soy Candle
  [12] sim=0.214  $  19.99  Bamboo Cutting Board
  [10] sim=0.148  $  22.00  Stainless Steel Water Bottle


## 7. Side by Side Comparison

You now compare pure vector search against hybrid search on the same query to see the difference clearly.

In [15]:
query = "affordable headphones under $50"

print(f"Query: '{query}'")
print()

# Pure vector search (no filtering)
print("PURE VECTOR SEARCH (no filters):")
vector_results = vector_search(query, top_k=5)
for r in vector_results:
    stock = "in stock" if r["in_stock"] else "OUT OF STOCK"
    flag = "  << over budget" if r["price"] > 50 else ""
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}  ({stock}){flag}")

print()

# Hybrid search (structured filters + vector ranking)
print("HYBRID SEARCH (SQL filters + vector ranking):")
hybrid_results = hybrid_search(query, verbose=False)
for r in hybrid_results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

print()
print("You can see that pure vector search returns expensive and out of stock items.")
print("Hybrid search enforces the constraints first, then ranks by relevance.")

Query: 'affordable headphones under $50'

PURE VECTOR SEARCH (no filters):
  [ 5] sim=0.476  $ 149.99  Noise Cancelling Headphones  (in stock)  << over budget
  [ 1] sim=0.432  $  29.99  Wireless Bluetooth Earbuds  (in stock)
  [ 3] sim=0.360  $  45.00  Portable Bluetooth Speaker  (in stock)
  [ 9] sim=0.250  $  95.00  Running Shoes  (OUT OF STOCK)  << over budget
  [ 4] sim=0.209  $  89.99  Mechanical Keyboard  (OUT OF STOCK)  << over budget

HYBRID SEARCH (SQL filters + vector ranking):
  [ 1] sim=0.473  $  29.99  Wireless Bluetooth Earbuds
  [ 3] sim=0.357  $  45.00  Portable Bluetooth Speaker
  [ 2] sim=0.192  $  12.99  USB-C Charging Cable 6ft

You can see that pure vector search returns expensive and out of stock items.
Hybrid search enforces the constraints first, then ranks by relevance.


## What You Learned

You now know how to build a hybrid retrieval system that combines structured outputs with vector search.

You stored embeddings directly in SQLite and used cosine similarity to rank results.

You used OpenAI structured outputs to reliably extract typed filters from natural language, without writing any parsing rules yourself.

You combined SQL filtering with vector ranking to get results that are both semantically relevant and exactly constrained.

This pattern scales well: swap SQLite for Postgres with pgvector or Supabase and you have a production system.

## Exercises With Solutions

For each exercise, read the prompt and then run the solution code directly below it.

### Exercise 1: Add a Rating Field

You will extend the product schema with a `rating` column and update the structured output to support a `min_rating` filter.

Tasks:
1. Run the solution cell below.
2. Observe how the filter now extracts a minimum rating from the query.
3. Notice how the SQL WHERE clause changes.

In [ ]:
# Solution for Exercise 1

# Add a rating column to the products table
cursor.execute("ALTER TABLE products ADD COLUMN rating REAL DEFAULT 4.0")

# Set some ratings
ratings = {
    1: 4.5, 2: 4.2, 3: 4.7, 4: 4.8, 5: 4.9,
    6: 4.0, 7: 3.8, 8: 4.3, 9: 4.6, 10: 4.4,
    11: 4.1, 12: 3.9, 13: 4.5, 14: 4.7, 15: 4.0
}
for product_id, rating in ratings.items():
    cursor.execute("UPDATE products SET rating = ? WHERE id = ?", (rating, product_id))
conn.commit()

# Extended filter schema
class ProductFilterV2(BaseModel):
    category: Optional[str] = None
    max_price: Optional[float] = None
    min_price: Optional[float] = None
    min_rating: Optional[float] = None
    in_stock_only: bool = True
    search_text: str

def parse_query_v2(user_query: str) -> ProductFilterV2:
    response = client.responses.parse(
        model=MODEL,
        instructions=(
            "You extract product search filters from a user query. "
            "Categories: electronics, clothing, home, food. "
            "Extract price constraints, category, stock preferences, and minimum rating. "
            "Ratings are on a 1 to 5 scale. "
            "Put the semantic search intent into search_text."
        ),
        input=user_query,
        text_format=ProductFilterV2,
    )
    return response.output_parsed

def build_sql_filter_v2(filters: ProductFilterV2) -> tuple[str, list]:
    conditions = []
    params = []
    if filters.category:
        conditions.append("category = ?")
        params.append(filters.category.lower())
    if filters.max_price is not None:
        conditions.append("price <= ?")
        params.append(filters.max_price)
    if filters.min_price is not None:
        conditions.append("price >= ?")
        params.append(filters.min_price)
    if filters.min_rating is not None:
        conditions.append("rating >= ?")
        params.append(filters.min_rating)
    if filters.in_stock_only:
        conditions.append("in_stock = 1")
    where_clause = " AND ".join(conditions) if conditions else "1=1"
    return where_clause, params

# Test it
query = "highly rated electronics under $100 with at least 4.5 stars"
filters = parse_query_v2(query)
print(f"Query: '{query}'")
print(f"Parsed: {filters}")
print()

where_clause, params = build_sql_filter_v2(filters)
print(f"SQL WHERE: {where_clause}")
print(f"Parameters: {params}")

sql = f"SELECT id, name, category, price, rating FROM products WHERE {where_clause}"
cursor.execute(sql, params)
print()
for row in cursor.fetchall():
    print(f"  [{row[0]:2d}] ${row[3]:>7.2f}  rating={row[4]}  {row[1]}")

### Exercise 2: Handle Empty Results Gracefully

You will build a version of hybrid search that automatically relaxes filters when no products match.

Tasks:
1. Run the solution cell below.
2. Notice the first query is too restrictive and returns zero results.
3. Observe how the fallback removes the price filter and retries.

In [ ]:
# Solution for Exercise 2

def hybrid_search_with_fallback(user_query: str, top_k: int = 5) -> list[dict]:
    """Hybrid search that relaxes filters if no results are found."""
    filters = parse_query(user_query)
    
    # Try with all filters
    candidates = sql_filter_search(filters)
    attempt = "all filters"
    
    if not candidates and (filters.max_price is not None or filters.min_price is not None):
        # Relax price constraint
        print(f"  No results with {attempt}. Removing price filter...")
        filters.max_price = None
        filters.min_price = None
        candidates = sql_filter_search(filters)
        attempt = "no price filter"
    
    if not candidates and filters.category:
        # Relax category constraint
        print(f"  No results with {attempt}. Removing category filter...")
        filters.category = None
        candidates = sql_filter_search(filters)
        attempt = "no category or price filter"
    
    if not candidates:
        print(f"  Still no results after relaxing all filters.")
        return []
    
    print(f"  Found {len(candidates)} candidates with {attempt}")
    
    # Vector rank
    query_embedding = get_embeddings([filters.search_text])[0]
    query_vec = np.array(query_embedding, dtype=np.float32)
    
    for c in candidates:
        c["similarity"] = cosine_similarity(query_vec, c["embedding"])
    
    candidates.sort(key=lambda x: x["similarity"], reverse=True)
    for c in candidates:
        del c["embedding"]
    
    return candidates[:top_k]

# Test: a query that is too specific
query = "electronics under $5"
print(f"Query: '{query}'")
results = hybrid_search_with_fallback(query)
print("Results:")
for r in results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

print()

# Test: a query that works on the first try
query = "food items under $15"
print(f"Query: '{query}'")
results = hybrid_search_with_fallback(query)
print("Results:")
for r in results:
    print(f"  [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

### Exercise 3: Multi Query Comparison

You will run several queries through all three retrieval approaches (pure vector, pure SQL, hybrid) and compare which approach gives the best results for each.

Tasks:
1. Run the solution cell below.
2. For each query, observe which approach returns the most useful results.
3. Identify one query where pure vector search fails and hybrid succeeds.

In [ ]:
# Solution for Exercise 3

test_queries = [
    "cheap snacks under $10",
    "premium audio equipment",
    "eco friendly home products under $25",
    "something warm to wear",
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("=" * 60)
    
    # Pure vector
    print("\n  VECTOR ONLY:")
    for r in vector_search(query, top_k=3):
        stock = "in stock" if r["in_stock"] else "OUT OF STOCK"
        print(f"    [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}  ({stock})")
    
    # Pure SQL (no vector ranking)
    print("\n  SQL ONLY (no semantic ranking):")
    filters = parse_query(query)
    filtered = sql_filter_search(filters)
    for p in filtered[:3]:
        print(f"    [{p['id']:2d}] ${p['price']:>7.2f}  {p['name']}")
    if not filtered:
        print("    (no matches)")
    
    # Hybrid
    print("\n  HYBRID (SQL + vector):")
    for r in hybrid_search(query, top_k=3, verbose=False):
        print(f"    [{r['id']:2d}] sim={r['similarity']:.3f}  ${r['price']:>7.2f}  {r['name']}")

## Summary

You built a complete hybrid retrieval system using three components:

1. **SQLite** for storing products and embeddings in one place.
2. **Structured outputs** for converting natural language into typed SQL filters.
3. **Vector search** for ranking filtered results by semantic similarity.

You now understand that hybrid retrieval is not a new invention. It is combining two well understood ideas (exact filtering and semantic ranking) into one pipeline. This is a pattern you can use in any project where users search with both constraints and intent.